# BÁO CÁO PHÂN TÍCH VÀ ĐÁNH GIÁ HIỆU NĂNG MÔ HÌNH HỒI QUY
# BỘ DỮ LIỆU: NYC TLC Yellow Taxi 02/2026 (yellow_tripdata_2026_02.csv)

---

### 1. Tổng quan bảng kết quả thực nghiệm
Dưới đây là bảng số liệu hiệu năng thu được từ ba mô hình hồi quy triển khai trên tập dữ liệu kiểm thử (Test Set):

| Mô hình | MAE | RMSE | $R^2$ |
| :--- | :---: | :---: | :---: |
| **OLS Cơ bản** | 6.470014 | 12.152525 | 0.695315 |
| **OLS Chọn biến** | 6.469775 | 12.152762 | 0.695303 |
| **Ridge Regression** | **6.467585** | **12.145806** | **0.695652** |

**Kết luận sơ bộ:** Mô hình tốt nhất theo tiêu chí tối đa hóa Hệ số xác định $R^2$ và tối thiểu hóa sai số (MAE, RMSE) là **Ridge Regression**.

### 2. Nhận xét và Giải thích chuyên sâu về các mô hình

#### 2.1. Tại sao kết quả giữa OLS Cơ bản và OLS Chọn biến lại gần như trùng khớp?
Thông thường, bước **OLS Chọn biến** (Feature Selection dựa trên p-value hoặc kiểm định VIF) được kỳ vọng sẽ loại bỏ bớt các biến nhiễu hoặc đa cộng tuyến để cải thiện mạnh hiệu năng. Tuy nhiên, kết quả thực nghiệm cho thấy sự chênh lệch về hiệu năng giữa hai mô hình này ở mức siêu nhỏ (lệch nhau từ số thập phân thứ 4 hoặc thứ 5).

* **Nguyên nhân hệ thống:** Hiện tượng này phản ánh **chất lượng xuất sắc của bước tiền xử lý dữ liệu (`DataPipeline`)** trước đó. Trong giai đoạn làm sạch dữ liệu, nhóm đã chủ động loại bỏ các biến dính lỗi cấu trúc dữ liệu nghiêm trọng như `improvement_surcharge` và `congestion_surcharge`. 
* Do các biến "độc hại" hoặc chứa đa cộng tuyến hoàn hảo đã bị triệt tiêu từ đầu, nên tập biến đi vào mô hình OLS cơ bản đã là một tập biến rất "sạch" và mang thông tin hữu ích. Khi thuật toán chọn biến thực hiện quét lại, nó không tìm thấy biến nào không có ý nghĩa thống kê để loại bỏ, dẫn đến cấu trúc ma trận thiết kế $X$ của hai mô hình gần như tương đương nhau.

#### 2.2. Tại sao Ridge Regression lại tốt nhất?
Mô hình Ridge đạt hiệu năng cao nhất trên cả 3 chỉ số ($MAE \downarrow, RMSE \downarrow, R^2 \uparrow$). Mặc dù mức tăng là nhỏ, nhưng điều này mang ý nghĩa toán học rất lớn:

* **Kiểm soát hiện tượng Đa cộng tuyến ẩn (Multicollinearity):** Mặc dù `DataPipeline` đã giải quyết đa cộng tuyến hoàn hảo (Perfect Multicollinearity), dữ liệu thực tế của Taxi luôn tồn tại đa cộng tuyến gần hoàn hảo (ví dụ: Quãng đường di chuyển `trip_distance` và Thời gian di chuyển `duration` tỷ lệ thuận rất mạnh với nhau). OLS cơ bản thường có xu hướng phóng đại các hệ số $\beta$ đối với các biến này, làm tăng phương sai của mô hình.
* **Cơ chế phạt $L_2$ Regularization:** Ridge Regression thêm một đại lượng phạt $\lambda \sum \beta_j^2$ vào hàm mất mát. Cơ chế này ép các hệ số hồi quy thu nhỏ lại (shrinkage) về gần 0, làm mượt mô hình và phân bổ đều tầm quan trọng cho các biến có liên quan thay vì phụ thuộc quá mức vào một biến. Kết quả là mô hình Ridge có **tính tổng quát hóa (Generalization)** tốt hơn, giảm hiện tượng Overfitting nhẹ và đạt kết quả tốt nhất khi kiểm thử trên dữ liệu lạ (Test Set).

#### 2.3. Ý nghĩa thực tế từ các chỉ số Sai số (MAE vs RMSE)
* **MAE $\approx$ 6.47:** Trung bình mỗi chuyến xe, mô hình dự đoán lệch khoảng **6.47 USD** so với thực tế.
* **RMSE $\approx$ 12.15:** RMSE phạt nặng các sai số lớn do có bình phương ($e^2$). Việc RMSE lớn gần gấp đôi MAE chỉ ra rằng trong tập dữ liệu tồn tại nhiều **Outliers chuyến xe bất thường** (ví dụ: các chuyến đi xuyên bang có cước phí cực lớn, hoặc các cuốc xe dính tắc đường hàng giờ liền khiến giá cước nhảy vọt ngoài quy luật tuyến tính thông thường của quãng đường).

In [ ]:
%pip install pandas

In [2]:
# Thống kê dữ liệu từ bộ dữ liệu trips yellow taxi tháng 02/2026
import pandas as pd

# Đọc file parquet
df = pd.read_parquet("./data/yellow_tripdata_2026-02.parquet")

print("--- KẾT QUẢ THỐNG KÊ THÁNG 02/2026 ---")
print(f"1. Tổng tiền trung bình (total_amount): ${df['total_amount'].mean():.2f}")
print(f"2. Quãng đường trung bình (trip_distance): {df['trip_distance'].mean():.2f} dặm")
print(f"3. Số hành khách trung bình (passenger_count): {df['passenger_count'].mean():.2f} người")
print(f"4. Tỉ lệ thanh toán bằng thẻ (Type 1): {(df['payment_type'] == 1).mean() * 100:.2f}%")
print(f"5. Tỉ lệ Store and Forward (N): {(df['store_and_fwd_flag'] == 'N').mean() * 100:.2f}%")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

### 3. Phân tích và đánh giá chuyên sâu về các dữ liệu trong bộ dữ liệu yellow_tripdata_2026_02.csv

#### 3.1. Thuế mà các hành khách phải trả (`mta_tax`)
* **Đánh giá:** Mức thuế MTA quy định cố định cho mỗi chuyến đi taxi truyền thống tại New York là **\$0.50** (có thể cộng thêm \$0.50 phụ phí quốc gia NY). 
* **Phân tích chuyên sâu:** Khi đặt con số \$0.50 này cạnh tổng hóa đơn trung bình lên tới **\$30.11**, tỷ trọng thuế chỉ chiếm vỏn vẹn **~1.66%**. Đây là một mức thuế **rất thấp**. Do tính chất cố định và chiếm tỷ lệ không đáng kể, khoản thuế này hoàn toàn không tác động đến tâm lý cân nhắc chi tiêu của hành khách, đồng thời đóng vai trò là một hằng số ổn định trong mô hình hồi quy.

#### 3.2. Cước phí bổ sung (`Surcharges`) so với cước gốc một chuyến đi
* **Đánh giá:** Tỷ lệ cước phí bổ sung (gồm `extra` giờ cao điểm/đêm, `improvement_surcharge` \$0.30 và đặc biệt là `congestion_surcharge` \$2.50 khu vực trung tâm Manhattan) ở mức **bình thường đến cao**.
* **Phân tích chuyên sâu:** Với tổng chi phí trung bình \$30.11 và quãng đường dài hơn 6 dặm, cước phí gốc (`fare_amount`) ước tính nằm trong khoảng \$20 - \$22. Khi đó, các khoản phụ phí (khoảng \$3.30 - \$4.30) chiếm tỷ lệ từ **15% đến 20%** hóa đơn. Đây là tỷ lệ đáng kể, cho thấy chính sách đô thị của NYC đang sử dụng công cụ kinh tế (phí kẹt xe) nhằm hạn chế lưu lượng xe taxi di chuyển không cần thiết tại trung tâm, tăng nguồn thu tái đầu tư vào hạ tầng giao thông công cộng.

#### 3.3. Cước phí Sân bay (`Airport Fee`) so với cước thông thường
* **Đánh giá:** Cước phí sân bay ở mức **Rất cao**.
* **Phân tích chuyên sâu:** Các chuyến đi hướng ra sân bay JFK (áp dụng giá phẳng Flat-rate cố định trên \$70) hoặc LaGuardia/Newark (tính theo mét nhưng kèm phí sân bay riêng biệt) đẩy tổng số tiền lên rất lớn. Quãng đường trung bình của toàn bộ tập dữ liệu đạt mức **6.24 dặm** (cao hơn mức thông thường từ 1-3 dặm của nội đô) chứng tỏ tháng 02/2026 ghi nhận một lượng lớn các chuyến đi liên quận dài hơi, đặc biệt là các chuyến đi hướng ra ngoại ô hoặc sân bay. Các chuyến xe này tạo ra các điểm dữ liệu có giá trị biên (Outliers doanh thu lớn), mang lại hiệu quả kinh tế vượt trội cho tài xế.

#### 3.4. Tỉ lệ các kiểu trả phí (`payment_type`)
* **Đánh giá:** Thanh toán bằng **Thẻ tín dụng (Credit Card - Type 1) chiếm ưu thế lớn với 60.35%**. Tỷ lệ còn lại thuộc về tiền mặt (Cash) và các hình thức khác.
* **Tại sao lại phổ biến?** Xu hướng thanh toán không tiền mặt đã trở thành văn hóa bắt buộc tại một trung tâm tài chính như New York. Hệ thống taxi Yellow tích hợp đầu đọc thẻ thông minh tiện lợi, tự động tính toán cước phí. Điều này không chỉ giúp việc chi trả diễn ra nhanh chóng mà còn giúp các doanh nhân dễ dàng nhận hóa đơn điện tử để xử lí công tác phí với công ty.

#### 3.5. Tỉ lệ `store_and_fwd_flag`
* **Đánh giá:** Chỉ số ghi nhận tỷ lệ `N` (No - Không lưu bộ nhớ đệm, truyền dữ liệu trực tiếp về máy chủ) đạt **69.84%**, đồng nghĩa với việc có **30.16%** chuyến xe được ghi nhận là `Y` (Yes - Lưu offline và đẩy lên server sau).
* **Phân tích chuyên sâu:** Tỷ lệ lưu trữ offline gần 30% là một con số **bất thường và khá cao** so với các năm trước (vốn thường >98% là trực tiếp). Điều này chỉ ra hiện tượng hạ tầng công nghệ hoặc mạng viễn thông di động (sóng 4G/5G) trên các xe taxi tại NYC trong giai đoạn này gặp trục trặc kỹ thuật, hoặc xe di chuyển qua các vùng lõm sóng, vùng hầm xuyên sông, hoặc hệ thống máy chủ trung tâm của TLC bị nghẽn mạch khiến thiết bị phần cứng trên xe phải bật chế độ lưu trữ đệm để bảo toàn dữ liệu hành trình.

#### 3.6. Đánh giá Tệp khách hàng qua Quãng đường trung bình (`trip_distance`)
* **Đánh giá:** Giá trị `trip_distance` trung bình đạt tới **6.24 dặm** là một con số **rất cao** đối với mô hình taxi đô thị.
* **Cấu trúc tệp khách hàng:** Quãng đường dài này phản ánh tệp khách hàng chủ lực của tháng là **tầng lớp trung lưu trở lên, doanh nhân, và khách du lịch có ngân sách lớn**. Khách hàng không chỉ di chuyển các chặng ngắn "last-mile" nội quận Manhattan nữa, mà họ sẵn sàng chi trả số tiền lớn để thực hiện các lộ trình liên quận (ví dụ: Manhattan sang Brooklyn, Queens) hoặc đi ra sân bay. Đây là tệp khách hàng **chịu chi tiền, ưu tiên tối đa sự thuận tiện và tốc độ di chuyển** thay vì lựa chọn tàu điện ngầm đường dài.

#### 3.7. Đánh giá Tệp khách hàng qua Số hành khách trung bình (`passenger_count`)
* **Đánh giá:** Số hành khách trung bình đạt **1.23 người/chuyến**.
* **Cấu trúc tệp khách hàng:** Chỉ số này khẳng định tệp khách hàng cốt lõi của Yellow Taxi là **những cá nhân** — chiếm tỷ trọng tuyệt đối. Xe taxi truyền thống phục vụ mục đích di chuyển cá nhân hóa cho giới công sở đi làm, đi gặp đối tác hoặc khách du lịch đơn lẻ. Yellow Taxi hoàn toàn không phải là phương tiện ưu tiên cho các nhóm gia đình hay hội nhóm đông người (những tệp thường sẽ đặt phân khúc xe lớn như UberXL).

#### 3.8. Phân tích dòng chảy lưu lượng theo Thời gian (`tpep_pickup_datetime`)
* **Đánh giá lưu lượng Ban ngày (6h - 18h) vs Ban đêm (18h - 6h sáng hôm sau):**
    * **Ban ngày (6AM - 6PM):** Chiếm tỷ trọng cao và ổn định. Đây là khoảng thời gian diễn ra guồng quay kinh tế chính của New York, bao gồm dòng người đi làm buổi sáng, di chuyển họp hành giữa các công ty và đỉnh điểm là làn sóng tan sở buổi chiều.
    * **Ban đêm (6PM - 6AM):** Mặc dù nửa đêm nhu cầu giảm, nhưng khung giờ từ 18h đến 22h lại là "mỏ vàng" nhờ nhu cầu giải trí, ăn uống tại các nhà hàng nghệ thuật và rạp hát Broadway tại khu vực Trung tâm (Midtown). Tuy nhiên, xét tổng thể cả chu kỳ 12 tiếng, lưu lượng ban ngày phục vụ di chuyển thương mại vẫn chiếm ưu thế áp đảo về số lượng chuyến đi (`trips count`).

#### 3.9. Đánh giá tính hợp lý của Tổng cước phí trung bình (`total_amount`)
* **Đánh giá:** Chi phí trung bình **\$30.11/chuyến** là mức phí **Hợp lý và tương xứng**.
* **Phân tích kinh tế:** Đối với một thị trường có chi phí sinh hoạt đắt đỏ như New York, mức chi trả ~\$30 cho một lộ trình dài hơn 6 dặm là hoàn toàn xứng đáng. Khách hàng không đơn thuần mua một dịch vụ vận tải cơ học, cái họ bỏ tiền ra mua chính là **sự tiết kiệm thời gian, tính an toàn, sự riêng tư và sự thoải mái tối đa** nhằm né tránh sự đông đúc của các phương tiện công cộng vào giờ cao điểm. Mức cước phí này đảm bảo được doanh thu tái đầu tư cho tài xế và phù hợp với túi tiền của tệp khách hàng cao cấp tại Manhattan.

### 4. Liên hệ thực tế: Ứng dụng của Data Fitting (Khớp dữ liệu)

**Data Fitting** là cốt lõi của việc chuyển đổi dữ liệu thô thành các quyết định kinh doanh tự động dựa trên toán học. Bằng cách tìm ra đường cong hoặc mặt phẳng tối ưu đi qua các điểm dữ liệu lịch sử, doanh nghiệp có thể xây dựng các hàm dự đoán thời gian thực.

#### Ví dụ cụ thể: Thuật toán Tính giá cước trước (Upfront Pricing) của các ứng dụng đặt xe (Grab, Uber, Taxi công nghệ)

Trong thực tế kinh doanh vận tải hành khách, việc hiển thị chính xác giá tiền trước khi khách hàng bấm "Đặt xe" là yếu tố sống còn để giữ chân người dùng. Quá trình áp dụng bài toán Data Fitting này diễn ra như sau:

1.  **Thu thập dữ liệu (Data Collection):** Hệ thống ghi nhận hàng triệu cuốc xe trong lịch sử với các thuộc tính: Thời điểm đặt xe (Giờ cao điểm/thấp điểm), Tọa độ đi/đến, Quãng đường vệ tinh dự kiến ($X_1$), Thời gian di chuyển ước tính dựa trên mật độ giao thông ($X_2$), Tình hình thời tiết ($X_3$).
2.  **Khớp dữ liệu (Data Fitting):** Doanh nghiệp sử dụng một mô hình hồi quy (tương tự như mô hình **Ridge** mà chúng ta vừa tìm ra) để fit vào ma trận dữ liệu này nhằm tìm ra bộ hệ số $\beta$ tối ưu. Hàm số cước phí có dạng:
    $$\text{Giá cước dự đoán} = \beta_0 + \beta_1 \cdot \text{Quãng đường} + \beta_2 \cdot \text{Thời gian} + \beta_3 \cdot \text{Hệ số mưa} + \dots$$
3.  **Vận hành thời gian thực (Inference):** Khi khách hàng mở ứng dụng và nhập điểm đi/đến, thuật toán không cần tính toán lại từ đầu mà chỉ cần lấy các thông số hiện tại nhân với bộ hệ số $\beta$ cố định đã học (Phép nhân ma trận $X \cdot \beta$ diễn ra trong vài mili-giây). Giá cước hiển thị ngay lập tức lên màn hình người dùng.

#### Tầm quan trọng của việc tối ưu hóa mô hình (như cách chọn Ridge thay vì OLS):
* Nếu dùng OLS cơ bản bị overfitting, khi gặp một ngày thời tiết xấu bất thường hoặc ngày lễ, mô hình có thể đưa ra mức giá "trên trời" (quá cao) khiến khách hàng hủy app, hoặc mức giá quá thấp khiến tài xế từ chối nhận cuốc xe, gây tổn thất doanh thu.
* Mô hình **Ridge Regression** ổn định các hệ số, giúp giá cước dao động một cách mượt mà, hợp lý, cân bằng được bài toán cung - cầu trong kinh doanh thực tế.

# KẾT THÚC PHÂN TÍCH VÀ ĐÁNH GIÁ